# Lab5.1 — Acceleratorul hardware Resize și Overlay-ul


În Lab4 am executat resize în software pe ARM. În Lab5 folosim un accelerator implementat în **Programmable Logic (PL)** și transferăm imaginea prin AXI DMA.


## 1. Arhitectura

Fluxul hardware este:

```text
DDR → AXI DMA (MM2S) → Resize accelerator → AXI DMA (S2MM) → DDR
```

Configurarea dimensiunilor se face din PS prin registre AXI-Lite.


Pregătim fișierele `.bit` și `.hwh` ale acceleratorului Resize pentru PYNQ-Z2.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

OVERLAY_DIR = Path("overlay")
OVERLAY_DIR.mkdir(exist_ok=True)
LOCAL_BIT = OVERLAY_DIR / "resizer.bit"
LOCAL_HWH = OVERLAY_DIR / "resizer.hwh"

def find_resizer_overlay():
    if LOCAL_BIT.exists() and LOCAL_HWH.exists():
        return LOCAL_BIT

    try:
        import pynq_helloworld
        pkg = Path(pynq_helloworld.__file__).resolve().parent / "notebooks" / "edge"
        pkg_bit = pkg / "resizer.bit"
        pkg_hwh = pkg / "resizer.hwh"
        if pkg_bit.exists() and pkg_hwh.exists():
            return pkg_bit
    except ImportError:
        pass

    urls = {
        LOCAL_BIT: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.bit",
        LOCAL_HWH: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.hwh",
    }
    for path, url in urls.items():
        if not path.exists():
            print(f"Downloading {path.name} ...")
            urlretrieve(url, path)

    return LOCAL_BIT

RESIZER_BIT = find_resizer_overlay()
print("Overlay:", RESIZER_BIT)

Încărcăm Overlay-ul în PL și verificăm că programarea FPGA-ului s-a realizat.


In [ ]:
from pynq import Overlay

overlay = Overlay(str(RESIZER_BIT))
print("Resize Overlay loaded.")

Afișăm IP-urile hardware descrise de fișierul HWH pentru a identifica DMA-ul și acceleratorul.


In [ ]:
for name, info in overlay.ip_dict.items():
    print(f"{name:25s} {info.get('type', '-')}")

Obținem obiectele Python asociate AXI DMA și acceleratorului Resize.


In [ ]:
dma = overlay.axi_dma_0
resizer = overlay.resize_accel_0

print("DMA     :", dma)
print("Resizer :", resizer)

Inspectăm registrele acceleratorului, prin care software-ul configurează rezoluția de intrare și de ieșire.


In [ ]:
print(resizer.register_map)

## 2. Registre importante

Acceleratorul primește patru parametri:

- `src_rows` — numărul de linii al imaginii sursă;
- `src_cols` — numărul de coloane al imaginii sursă;
- `dst_rows` — numărul de linii al imaginii rezultate;
- `dst_cols` — numărul de coloane al imaginii rezultate.


## 3. Exercițiu

Din `overlay.ip_dict`, identificați tipul și adresa fizică pentru `axi_dma_0` și `resize_accel_0`.


Afișați informațiile cerute direct din `ip_dict`, fără adrese hard-codate.


In [ ]:
# Scrieți aici soluția


## 4. Întrebări

1. Ce rol are AXI DMA?
2. Ce rol are AXI-Lite în acest design?
3. Unde rulează codul Python și unde rulează operația Resize?
4. De ce avem nevoie atât de `.bit`, cât și de `.hwh`?
